# Week 8 — GPSat: radar freeboard along-track (Assignment Example 1)

**Task:** Adapt the along-track notebook to interpolate **radar freeboard** (not sea level / SLA).

This notebook is trimmed for **Google Colab**: install dependencies, clone `GPSat`, then run the GPOD along-track section. Upload your GPOD `*v1.proc` files under a folder you set below (e.g. Google Drive).


In [ ]:

# Optional: mount Google Drive if GPOD data lives there
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not on Colab — skip Drive mount.")


In [ ]:
!pip install -q gpflow deprecated dataclasses-json cartopy


In [ ]:

import os
import subprocess
import sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    work_dir = "/content"
    os.chdir(work_dir)
    repo_dir = os.path.join(work_dir, "GPSat")
    if not os.path.isdir(repo_dir):
        subprocess.run(
            ["git", "clone", "https://github.com/CPOMUCL/GPSat.git"],
            check=True,
        )
    os.chdir(repo_dir)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
    sys.path.insert(0, "/content/GPSat")
else:
    print("Running locally — ensure GPSat is on PYTHONPATH.")


## 1. Along-track radar freeboard with `LocalExpertOI`

We load GPOD L2I-style text tracks, take **radar freeboard** (column index 4), train the GP on **lead** observations only, predict at all along-track points, then compare to **linear interpolation** of lead freeboard along distance.

**Column reference (GPOD):** index 4 = radar freeboard; 7 = class (1=lead, 2=floe); 9 = raw elevation; 10 = MSS; etc.


In [ ]:

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeat

from datetime import datetime, timedelta

from GPSat import get_parent_path
from GPSat.utils import WGS84toEASE2_New
from GPSat.local_experts import LocalExpertOI
from GPSat.postprocessing import smooth_hyperparameters, glue_local_predictions_2d
import GPSat

# ---------------------------------------------------------------------------
# EDIT THIS: folder(s) containing satellite subfolders with *v1.proc tracks
# Colab example:
# GPOD_ROOT = "/content/drive/MyDrive/AI4EO/Week 8/GPOD"
# ---------------------------------------------------------------------------
GPOD_ROOT = "/content/drive/MyDrive/AI4EO/Week 8/GPOD"


def GCdist(X, Xs):
    lon1 = np.radians(X.T[(0,)].T)
    lat1 = np.radians(X.T[(1,)].T)
    lon2 = np.radians(Xs.T[(0,)].T)
    lat2 = np.radians(Xs.T[(1,)].T)
    r = 6356752.0
    Clat1 = np.cos(lat1)
    Clat2 = np.cos(lat2)
    Slat1 = np.sin(lat1)
    Slat2 = np.sin(lat2)
    Clon1 = np.cos(lon1)
    Clon2 = np.cos(lon2)
    Slon1 = np.sin(lon1)
    Slon2 = np.sin(lon2)
    n1 = np.array([Clat1 * Clon1, Clat1 * Slon1, Slat1]).T
    n2 = np.array([Clat2 * Clon2, Clat2 * Slon2, Slat2]).T
    return r * np.arccos(np.dot(n1, n2.T))


def process_track(track, grab_dates, satellite, count):
    # Process a single track file if it falls on a date of interest.
    date = track.split("/")[-1].split("T")[0].split("_")[-1]
    if date not in grab_dates:
        return count, None
    time = 14975 + (datetime.strptime(date, "%Y%m%d") - datetime(2011, 1, 1)).days
    f = np.genfromtxt(track)

    ID = np.where((f[:, 7] < 3) & (f[:, 11] >= 0.75))[0]
    classes = np.where(f[ID, 7] == 1.0, "lead", "floe")

    elvmss = f[ID, 9] - f[ID, 10]
    elvmss[classes == "floe"] -= 0.1626

    xg, yg = WGS84toEASE2_New(f[ID, 0], f[ID, 1])
    N = len(xg)

    track_df = pd.DataFrame()
    track_df["x"] = xg
    track_df["y"] = yg
    track_df["t"] = [time] * N
    track_df["lon"] = f[ID, 0]
    track_df["lat"] = f[ID, 1]
    track_df["date"] = [date] * N
    track_df["class"] = classes
    track_df["interpSLA"] = f[ID, 2]
    track_df["elevation"] = elvmss
    track_df["radar_freeboard"] = f[ID, 4]
    track_df["SAT"] = [satellite] * N
    track_df["track"] = [count] * N
    return count + 1, track_df


interp_date = datetime(2019, 1, 15)
grab_dates = [(interp_date - timedelta(days=x)).strftime("%Y%m%d") for x in range(5)] + [
    (interp_date + timedelta(days=x)).strftime("%Y%m%d") for x in range(1, 5)
]

dirs = sorted(glob.glob(os.path.join(GPOD_ROOT, "*")))
track_dfs = []
count = 0
for directory in dirs:
    satellite = os.path.basename(directory)
    if satellite in {"SARIN", "SAR"}:
        satellite = "CS2"
    for track in sorted(glob.glob(os.path.join(directory, "*v1.proc"))):
        count, track_df = process_track(track, grab_dates, satellite, count)
        if track_df is not None:
            track_dfs.append(track_df)

if not track_dfs:
    raise FileNotFoundError(
        f"No tracks found under {GPOD_ROOT!r}. Set GPOD_ROOT and add *v1.proc files."
    )

df = pd.concat(track_dfs, ignore_index=True)
df


In [ ]:

interptrack = 1
df_val = df.loc[df["track"] == interptrack].copy()

fig, ax = plt.subplots(1, figsize=(5, 5), subplot_kw=dict(projection=ccrs.NorthPolarStereo()))
ax.set_extent([-180, 180, 65, 90], ccrs.PlateCarree())
ax.add_feature(cfeat.LAND, color=(0.7, 0.7, 0.7))
sc = ax.scatter(
    df_val["lon"],
    df_val["lat"],
    s=0.5,
    c=df_val["radar_freeboard"],
    cmap="viridis",
    vmin=0,
    vmax=0.5,
    transform=ccrs.PlateCarree(),
)
plt.colorbar(sc, ax=ax, shrink=0.6, label="Radar freeboard (m)")
plt.show()
df_val.head()


In [ ]:

leads = np.where(df_val["class"] == "lead")[0]
floes = np.where(df_val["class"] == "floe")[0]
fig, ax = plt.subplots(1, figsize=(15, 5))
ax.scatter(leads, df_val["radar_freeboard"].iloc[leads], color="b", s=1, label="leads")
ax.scatter(floes, df_val["radar_freeboard"].iloc[floes], color="k", s=1, label="floes")
ax.set_ylabel("Radar freeboard (m)")
ax.set_xlabel("Along-track index")
ax.legend()
plt.show()


In [ ]:

Xs = np.array([df_val["lon"], df_val["lat"]]).T
r_exp = GCdist(Xs, Xs)
df_val["dist_along_track"] = r_exp[0, :]
exp_grid = np.arange(0, np.sum(r_exp[0, :]), 100e3)
locs = []
for ix in exp_grid:
    temp = np.abs(r_exp[0, :] - ix)
    dm = np.where(temp == np.min(temp))
    if len(locs) <= 16:
        locs.append(dm[0][0])

fig, ax = plt.subplots(
    1, 2, figsize=(10, 5), subplot_kw=dict(projection=ccrs.NorthPolarStereo())
)
for c in range(2):
    ax[c].set_extent([-180, 180, 65, 90], ccrs.PlateCarree())
    ax[c].add_feature(cfeat.LAND, color=(0.7, 0.7, 0.7))
    if c == 0:
        ax[c].scatter(
            df_val["lon"],
            df_val["lat"],
            c=df_val["radar_freeboard"],
            s=0.5,
            cmap="viridis",
            vmin=0,
            vmax=0.5,
            transform=ccrs.PlateCarree(),
        )
    else:
        ax[c].scatter(
            df_val["lon"].iloc[locs],
            df_val["lat"].iloc[locs],
            s=5,
            c="k",
            transform=ccrs.PlateCarree(),
        )
plt.show()

experts = df_val.iloc[locs]
experts


In [ ]:

store_path = get_parent_path(
    "/content/drive/MyDrive/AI4EO/Week 8/", "GPSat_alongtrack_radar_freeboard_3D.h5"
)

if os.path.exists(store_path):
    print(f"Removing old file: {store_path}")
    os.remove(store_path)

data = {
    "data_source": df.loc[
        (df["class"] == "lead")
        & np.isfinite(df["radar_freeboard"])
        & (np.abs(df["radar_freeboard"]) < 2.0)
    ],
    "obs_col": "radar_freeboard",
    "coords_col": ["x", "y", "t"],
    "local_select": [
        {"col": ["x", "y"], "comp": "<=", "val": 100_000},
        {"col": "t", "comp": "<=", "val": 2},
        {"col": "t", "comp": ">=", "val": -2},
    ],
    "global_select": [{"col": "lat", "comp": ">=", "val": 45}],
}

local_expert = {"source": experts}

pred_loc = {
    "method": "from_dataframe",
    "df": df_val,
    "max_dist": 200_000,
}

model = {
    "oi_model": "GPflowGPRModel",
    "init_params": {
        "coords_scale": [10000, 10000, 1],
        "minibatch_size": 100,
        "num_inducing_points": 200,
    },
    "constraints": {
        "lengthscales": {
            "low": [100_000, 100_000, 1e-08],
            "high": [200000, 200000, 4],
        },
        "likelihood_variance": {"low": 0.001, "high": 10},
    },
}

locexp = LocalExpertOI(
    expert_loc_config=local_expert,
    data_config=data,
    model_config=model,
    pred_loc_config=pred_loc,
)

locexp.run(store_path=store_path, optimise=True)

smooth_configs = {
    "lengthscales": dict(l_x=200_000, l_y=200_000, max=12),
    "likelihood_variance": dict(l_x=200_000, l_y=200_000),
    "kernel_variance": dict(l_x=200_000, l_y=200_000, max=0.1),
}

smooth_hyperparameters(
    result_file=store_path,
    params_to_smooth=["lengthscales", "kernel_variance", "likelihood_variance"],
    smooth_config_dict=smooth_configs,
    save_config_file=False,
)

model["load_params"] = {"file": store_path, "table_suffix": "_SMOOTHED"}
locexp = LocalExpertOI(
    expert_loc_config=local_expert,
    data_config=data,
    model_config=model,
    pred_loc_config=pred_loc,
)

locexp.run(store_path=store_path, optimise=False, predict=True, table_suffix="_SMOOTHED")

import shutil

if shutil.which("ptrepack"):
    tmp = store_path + ".tmp"
    os.system(f"ptrepack --chunkshape=auto --propindexes --complib=blosc {store_path} {tmp}")
    os.replace(tmp, store_path)
else:
    print("ptrepack not found — skipping HDF5 repack.")

print("Done.")


In [ ]:

dfs, _ = GPSat.local_experts.get_results_from_h5file(store_path)
preds_data = dfs["preds_SMOOTHED"]
preds_data.head()


In [ ]:

inference_radius = 200_000
plt_data = glue_local_predictions_2d(
    preds_df=preds_data,
    pred_loc_cols=["pred_loc_x", "pred_loc_y"],
    xprt_loc_cols=["x", "y"],
    vars_to_glue=["f*", "f*_var"],
    inference_radius=inference_radius,
)
plt_data.head()


In [ ]:

fb_df = pd.merge(
    df_val[
        ["lon", "lat", "x", "y", "radar_freeboard", "dist_along_track", "class"]
    ],
    plt_data[["pred_loc_x", "pred_loc_y", "f*"]],
    left_on=["x", "y"],
    right_on=["pred_loc_x", "pred_loc_y"],
)

d = fb_df["dist_along_track"].values
lm = fb_df["class"].values == "lead"
leads_d = d[lm]
leads_fb = fb_df.loc[lm, "radar_freeboard"].values
order = np.argsort(leads_d)
leads_d = leads_d[order]
leads_fb = leads_fb[order]
fb_df["linear_freeboard"] = np.interp(d, leads_d, leads_fb)
fb_df["freeboard_diff"] = fb_df["linear_freeboard"] - fb_df["f*"]
fb_df


In [ ]:

leads_df = fb_df[fb_df["class"] == "lead"]
floes_df = fb_df[fb_df["class"] == "floe"]

fig, ax = plt.subplots(1, figsize=(15, 5))
ax.grid(True, alpha=0.5)
ax.scatter(
    leads_df["dist_along_track"] / 1000,
    leads_df["radar_freeboard"],
    color="b",
    s=1,
    label="leads (obs)",
)
ax.scatter(
    floes_df["dist_along_track"] / 1000,
    floes_df["radar_freeboard"],
    color="k",
    s=1,
    label="floes (obs)",
)
ax.plot(
    fb_df["dist_along_track"] / 1000,
    fb_df["linear_freeboard"],
    color="b",
    lw=1,
    label="Linear interp (leads)",
)
ax.plot(
    fb_df["dist_along_track"] / 1000,
    fb_df["f*"],
    color="r",
    lw=1,
    label="GPSat GP",
)
ax.set_ylim(-0.05, 0.6)
ax.set_ylabel("Radar freeboard (m)")
ax.set_xlabel("Distance along track (km)")
ax.legend()
plt.show()


In [ ]:

fig, axs = plt.subplots(
    1, 4, figsize=(14, 5), subplot_kw=dict(projection=ccrs.NorthPolarStereo())
)
axs[0].set_extent([-180, 180, 71, 90], ccrs.PlateCarree())
axs[0].add_feature(cfeat.OCEAN, color="lightgrey")
axs[0].add_feature(cfeat.LAND, color="grey")
m0 = axs[0].scatter(
    fb_df["lon"],
    fb_df["lat"],
    s=0.5,
    c=fb_df["linear_freeboard"],
    cmap="viridis",
    vmin=0,
    vmax=0.5,
    transform=ccrs.PlateCarree(),
)
axs[0].set_title("Linear interp (leads)")
fig.colorbar(m0, ax=axs[0], label="m", orientation="horizontal", pad=0.01)

axs[1].set_extent([-180, 180, 71, 90], ccrs.PlateCarree())
axs[1].add_feature(cfeat.OCEAN, color="lightgrey")
axs[1].add_feature(cfeat.LAND, color="grey")
m1 = axs[1].scatter(
    fb_df["lon"],
    fb_df["lat"],
    s=0.5,
    c=fb_df["f*"],
    cmap="viridis",
    vmin=0,
    vmax=0.5,
    transform=ccrs.PlateCarree(),
)
axs[1].set_title("GPSat radar freeboard")
fig.colorbar(m1, ax=axs[1], label="m", orientation="horizontal", pad=0.01)

axs[2].set_extent([-180, 180, 71, 90], ccrs.PlateCarree())
axs[2].add_feature(cfeat.OCEAN, color="lightgrey")
axs[2].add_feature(cfeat.LAND, color="grey")
m2 = axs[2].scatter(
    fb_df["lon"],
    fb_df["lat"],
    s=0.5,
    c=fb_df["freeboard_diff"],
    cmap="RdBu_r",
    vmin=-0.1,
    vmax=0.1,
    transform=ccrs.PlateCarree(),
)
axs[2].set_title("Linear − GPSat")
fig.colorbar(m2, ax=axs[2], label="m", orientation="horizontal", pad=0.01, extend="both")

axs[3].set_extent([-180, 180, 71, 90], ccrs.PlateCarree())
axs[3].add_feature(cfeat.OCEAN, color="lightgrey")
axs[3].add_feature(cfeat.LAND, color="grey")
m3 = axs[3].scatter(
    fb_df["lon"],
    fb_df["lat"],
    s=0.5,
    c=fb_df["dist_along_track"] / 1000,
    cmap="viridis",
    transform=ccrs.PlateCarree(),
)
axs[3].set_title("Along-track distance")
fig.colorbar(m3, ax=axs[3], label="km", orientation="horizontal", pad=0.01)

plt.show()


In [ ]:

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(fb_df["dist_along_track"] / 1000, fb_df["freeboard_diff"])
ax.axhline(y=0, color="k", linestyle="--", alpha=0.8)
ax.set_xlabel("Distance along track (km)")
ax.set_ylabel("Linear − GPSat (m)")
ax.set_title("Difference along track")
plt.show()
